# C9-dimensionality-reduction — Session 4: Maps and Structure — What a 2-D Picture Can and Cannot Tell You

*One class session, roughly 85 minutes. Builds on Sessions 1–3 (PCA as
the SVD route, projections), C8-embeddings (nearest neighbors,
`argsort` ranking), and F1-scientific-python (broadcasting,
matplotlib).*

**This session:** the limits of *linear* maps — a seeded
**curved-manifold dataset** (a ribbon bent into an S) on which the PCA
view keeps $90\%$ of the variance and still merges far-apart points;
the **neighbor-graph idea** behind nonlinear map-making methods (the
exam's named example is UMAP), taught in this course's *stated-fact*
register — what such maps preserve (local neighborhoods) and what they
distort (global distances), with **no** map-making library run here;
a computable **local-structure metric** (the fraction of $k$ nearest
neighbors a view preserves); and the discipline of
**local-vs-global-structure** — which questions each kind of 2-D view
can honestly answer.

Try every checkpoint by hand first, then verify in code.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. A Dataset That Bends: the S-Ribbon

**Motivation.**
Session 1's cluster blobs were kind to PCA: their structure *was* their
variance.
The stress test for any 2-D map is data whose true shape is **curved**
— points living on a bent surface inside a higher-dimensional space.
On such data, "flatten me faithfully" and "keep the big distances
honest" become *competing* goals, and every map must pick a side.
That competition is this session's subject.

**The construction (seeded, NumPy only).**
Take a parameter $t \in [-\tfrac{3\pi}{2}, \tfrac{3\pi}{2}]$ and bend
the interval into an S in the $xz$-plane:
$$x = \sin t, \qquad z = \operatorname{sign}(t)\,(\cos t - 1),$$
then extrude it into a ribbon with a width coordinate
$y \sim \mathrm{U}(0, 6)$ and add a little noise.
One property is engineered on purpose: the curve has **unit speed**
($x'^2 + z'^2 = \cos^2 t + \sin^2 t = 1$), so $|\Delta t|$ between two
points *is* their distance along the ribbon.
`t` is a ruler laid along the bent sheet — which is exactly what a
faithful flattening should recover.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)

n = 600
t = np.sort(rng.uniform(-1.5 * np.pi, 1.5 * np.pi, n))   # position along the ribbon
y = rng.uniform(0.0, 6.0, n)                              # width coordinate
x = np.sin(t)
z = np.sign(t) * (np.cos(t) - 1.0)
X3 = np.column_stack([x, y, z]) + rng.normal(0.0, 0.03, (n, 3))

print("X3 shape:", X3.shape)
print("column variances:", np.round(X3.var(axis=0, ddof=1), 4))

fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2))
sc = axes[0].scatter(X3[:, 0], X3[:, 2], c=t, s=8, cmap="viridis")
axes[0].set_xlabel("x"); axes[0].set_ylabel("z")
axes[0].set_title("Side view (x, z): the S profile")
axes[1].scatter(X3[:, 1], X3[:, 0], c=t, s=8, cmap="viridis")
axes[1].set_xlabel("y (width)"); axes[1].set_ylabel("x")
axes[1].set_title("Front view (y, x): the ribbon's extent")
fig.colorbar(sc, ax=axes, label="t (position along the ribbon)", shrink=0.9)
plt.show()

The side view shows the profile: an S whose two hooks curl back toward
the middle.
The color ramp is the ruler — follow it from purple ($t = -4.7$) to
yellow ($t = +4.7$) and you walk the whole ribbon, length
$3\pi \approx 9.42$.
The variances ($0.51$, $3.03$, $2.00$) already hint at what a
variance-chasing method will do: the *width* direction $y$ carries the
most variance, the bend $z$ comes second, and the poor $x$ axis — the
direction that makes the S an S — carries the least.

### Checkpoint 1

1. Verify by differentiation that the profile curve has unit speed for
   $t \neq 0$ (so $|\Delta t|$ = distance along the curve).
2. Two points sit at $t = 1.0$ and $t = 4.0$ with the same $y$.
   Roughly how far apart are they *along the ribbon*?
3. Which of the three coordinates would you delete if you wanted to
   *hide* the S shape, and why?

## 2. The PCA View: 90% of the Variance, Folded Branches

Sessions 1–2's recipe, verbatim: center, SVD, keep two scores
columns.

In [ ]:
mu = X3.mean(axis=0)
Xc = X3 - mu
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
evr = s**2 / (s**2).sum()
print("evr:", np.round(evr, 4), "| top-2 keep:", np.round(evr[:2].sum(), 4))
print("v1:", np.round(Vt[0], 4))
print("v2:", np.round(Vt[1], 4))
print("v3 (dropped):", np.round(Vt[2], 4))

P = Xc @ Vt[:2].T                                  # the 2-D PCA view

fig, ax = plt.subplots(figsize=(6.6, 4.6))
sc = ax.scatter(P[:, 0], P[:, 1], c=t, s=8, cmap="viridis")
ax.set_xlabel("PC1 score"); ax.set_ylabel("PC2 score")
ax.set_title("PCA view of the ribbon (colored by position t)")
fig.colorbar(sc, label="t")
plt.tight_layout()
plt.show()

The numbers first: the top-2 plane keeps $90.91\%$ of the variance,
and the direction it drops — $v_3 \approx (-0.999, 0.004, -0.042)$ —
is essentially the $x$ axis, exactly as the column variances predicted.
PC1 is essentially width, PC2 essentially the bend.

Now the picture: the color ramp is *broken*.
Instead of one smooth sweep of color across the sheet, yellow-green
points sit **on top of** blue points: squashing out $x$ has pressed
different stretches of the ribbon onto each other (in the side view,
imagine looking at the S from directly above — the hooks land on the
middle).
This is Session 1 §6's warning, cashed: **variance kept is not
structure kept.**
A $91\%$ evr sounds like a faithful picture; for *neighborhood*
questions, this one lies.

### Checkpoint 2

1. Read $v_3$'s entries: why does dropping (essentially) $x$ fold the
   S, in terms of the side-view picture?
2. The evr of the *width* direction is the largest single share. Is
   ribbon width interesting structure here? What does that say about
   trusting evr to rank "importance"?
3. Session 1 §6's clusters survived projection almost perfectly. Name
   the property those blobs had that this ribbon lacks.

## 3. False Neighbors, Measured

"The view lies" becomes a checkable claim with one search: **find the
pair of points closest together in the PCA view among pairs that are
far apart along the ribbon** ($|\Delta t| > 2$).
Broadcast the view's squared distances, mask the along-ribbon-near
pairs to $\infty$, take the argmin.

In [ ]:
D2_view = ((P[:, None, :] - P[None, :, :]) ** 2).sum(axis=2)     # (600, 600)
dt = np.abs(t[:, None] - t[None, :])
candidates = np.where(dt > 2.0, D2_view, np.inf)
i, j = np.unravel_index(np.argmin(candidates), candidates.shape)

d_view = np.sqrt(D2_view[i, j])
d_3d = np.sqrt(((X3[i] - X3[j]) ** 2).sum())
print(f"points #{i} and #{j}:")
print(f"  positions along the ribbon: t = {t[i]:.3f} and t = {t[j]:.3f}  (|dt| = {dt[i, j]:.3f})")
print(f"  distance in the PCA view : {d_view:.4f}")
print(f"  distance in 3-D          : {d_3d:.4f}")

Points $27$ and $177$ sit $2.558$ apart along the ribbon and $1.9532$
apart in 3-D — yet the PCA view places them $0.0231$ apart:
**false neighbors**, manufactured by the projection.

One direction of error is impossible, though, and it is worth proving
on the spot: a PCA view is a projection onto orthonormal axes, so it
can *shorten* distances but never stretch them (Session 1, Checkpoint
6.3).
Numerically:

In [ ]:
D_orig = np.sqrt(((X3[:, None, :] - X3[None, :, :]) ** 2).sum(axis=2))
D_view = np.sqrt(D2_view)
print("max over all pairs of (view distance - true distance):",
      np.max(D_view - D_orig))

The maximum is $0.0$ — no pair, out of all $\binom{600}{2}$, gets
even a hair farther apart in the view.
A projection **understates or tells the truth; it never exaggerates.**
So a PCA view is trustworthy one-sidedly: points far apart *in the
view* really are far apart; points close in the view might not be.

### Checkpoint 3

1. State the one-sided guarantee precisely: which of "close in view"
   / "far in view" is evidence, and evidence of what?
2. Why did the search mask pairs with $|\Delta t| \le 2$ instead of
   just taking the overall closest view pair?
3. Could a pair be *close in 3-D* but far apart along the ribbon?
   Where on the S would you look for one?

## 4. Unrolling the Ribbon, and the Neighbor-Graph Idea

We built this ribbon, so we own the perfect flattening: plot each point
at $(t, y)$ — position along the sheet, position across it.
Call it the **unrolled view**.

In [ ]:
Uview = np.column_stack([t, y])                    # the unrolled coordinates

fig, ax = plt.subplots(figsize=(6.6, 3.4))
sc = ax.scatter(Uview[:, 0], Uview[:, 1], c=t, s=8, cmap="viridis")
ax.set_xlabel("t (along the ribbon)"); ax.set_ylabel("y (across)")
ax.set_title("The unrolled view: the ribbon flattened by its own recipe")
fig.colorbar(sc, label="t")
plt.tight_layout()
plt.show()

A clean rectangle, the color ramp unbroken: neighborhoods on the sheet
stay neighborhoods on the page.
No linear projection can produce this picture — unbending an S is not a
matrix multiplication.

**The catch:** we could only unroll because we *built* the data and
kept the recipe ($t$).
Real data arrives with no recipe.
**Neighbor-graph map-making methods** — the exam's named example is
**UMAP** — aim to recover an unrolling from the data alone, and this
course teaches them as a *concept*, in the stated-fact register (no
map-making library is run in this unit):

- **The mechanism, in one breath:** connect each point to its $k$
  nearest neighbors in the original space, forming a **neighbor
  graph**; then arrange points in 2-D so that graph neighbors land
  close together, letting non-neighbors fall where they may.
  The graph is the only input — and a graph on the S-ribbon connects
  points *along the sheet* (tiny within-sheet distances beat
  across-the-fold chords), so a good layout of it approximates our
  unrolled rectangle.
- **Stated fact 1 — local first.** Such methods *prioritize preserving
  local neighborhood structure over global distances*: who your
  neighbors are is (approximately) kept; how far you sit from distant
  regions is not.
- **Stated fact 2 — global distances are casualties.** On-page
  distances between far-apart groups, apparent densities, and apparent
  cluster sizes carry no quantitative meaning.
- **Stated fact 3 — the axes mean nothing.** The layout can be
  rotated, reflected, or re-run into a different-looking picture; its
  axes are not feature recipes (contrast PCA, where each axis is a
  stated unit-vector blend of the original features), and they carry
  **no units**.
- **Stated fact 4 — knobs and randomness.** The result depends on $k$,
  on other settings, and on random initialization; two honest runs can
  disagree in layout while agreeing in neighborhood content.

The unrolled view is our stand-in for "the picture such a method is
trying to draw" — which lets us *measure* the trade-offs, next.

### Checkpoint 4

1. Why does a $k$-NN graph on the S-ribbon essentially never contain a
   fold-crossing edge? (Compare typical within-sheet neighbor
   distances to the $\approx 1.9$ chord of Section 3's false pair —
   with $600$ points on a $9.4 \times 6$ sheet.)
2. Which stated fact says a UMAP-style picture cannot answer "is
   group A twice as far from B as from C"?
3. Give one way PCA axes are *more* interpretable than
   neighbor-graph-layout axes.

## 5. A Local-Structure Metric You Can Compute

"Preserves local structure" should be a number, not a slogan.
The metric this course pins — used by the practice set — is
**$k$-NN preservation**:

> For each point, take its $k$ nearest neighbors in the original
> space and its $k$ nearest in the view; record the fraction shared;
> average over points.

$1.0$ means every neighborhood survived; $k/(n-1) \approx 0.017$ is
chance level.
The distance matrices broadcast (the register from C8); the per-point
overlap count is list bookkeeping, where a Python loop is allowed.

In [ ]:
def knn_indices(A, k):
    # row i: indices of A's k nearest rows to row i (self excluded)
    sq = (A * A).sum(axis=1)
    D2 = np.maximum(sq[:, None] + sq[None, :] - 2.0 * (A @ A.T), 0.0)
    np.fill_diagonal(D2, np.inf)
    return np.argsort(D2, axis=1)[:, :k]

def knn_preservation(A_orig, A_view, k):
    nb_o = knn_indices(A_orig, k)
    nb_v = knn_indices(A_view, k)
    fracs = np.empty(len(A_orig))
    for idx in range(len(A_orig)):                     # bookkeeping loop
        fracs[idx] = np.intersect1d(nb_o[idx], nb_v[idx]).size / k
    return fracs.mean()

k = 10
P1 = Xc @ Vt[:1].T                                    # 1-D PCA view, for scale
for name, view in [("unrolled (t, y)", Uview), ("PCA top-2", P), ("PCA top-1", P1)]:
    print(f"  k-NN preservation, {name:15s}: {knn_preservation(X3, view, k):.4f}")

A clean ladder:

- **unrolled $0.9322$** — nearly every neighborhood intact (the $7\%$
  shortfall is the noise jiggling near-ties at neighborhood edges);
- **PCA top-2 $0.5877$** — the fold destroys four neighborhoods in
  ten, despite the $91\%$ evr;
- **PCA top-1 $0.1303$** — a 1-D shadow of a 2-D sheet keeps little
  beyond chance.

One metric, one story: *for this bent dataset*, the map built from
neighborhood information beats the map built from variance — **at the
local game**.
The global game inverts the ranking, and that is the next section.

### Checkpoint 5

1. Why must the self-index be excluded (the `np.inf` diagonal) before
   the argsort?
2. For $n = 600$, $k = 10$: what preservation would a *random* 2-D
   scatter score, roughly, and why?
3. The unrolled view scores $0.93$, not $1.00$, even though it is the
   ribbon's own recipe. What eats the last $7\%$?

## 6. The Global Metric, and Which Questions Each View Answers

The mirror-image metric: **worst-case stretch** — over all pairs, the
largest ratio of view distance to true 3-D distance.
A faithful-at-a-distance view keeps it near $1$.

In [ ]:
D_unr = np.sqrt(((Uview[:, None, :] - Uview[None, :, :]) ** 2).sum(axis=2))
iu = np.triu_indices(n, 1)                       # each pair once

print("worst-case stretch, PCA view     :", np.round(np.max(D_view[iu] / D_orig[iu]), 4))
print("worst-case stretch, unrolled view:", np.round(np.max(D_unr[iu] / D_orig[iu]), 4))

a, b = 0, n - 1                                  # the ribbon's two ends
print(f"the two ends of the S: 3-D distance {D_orig[a, b]:.4f}, "
      f"unrolled distance {D_unr[a, b]:.4f}  (ratio {D_unr[a, b] / D_orig[a, b]:.3f})")

flat = np.argmax(D_unr[iu] / D_orig[iu])
ia, jb = iu[0][flat], iu[1][flat]
print(f"worst pair: t = {t[ia]:.3f} vs t = {t[jb]:.3f}: "
      f"3-D {D_orig[ia, jb]:.4f}, unrolled {D_unr[ia, jb]:.4f}")

The PCA view's stretch is $1.0$ — the projection guarantee of
Section 3, now as a metric.
The unrolled view stretches its worst pair $4.3\times$: a hook point
($t = -4.658$) and a middle point ($t = 0.362$) sit only $1.1667$
apart in 3-D — the hook curls back toward the middle — but the
unrolling tears them $5.0204$ apart.
The S's two ends tell the same story at the largest scale: $3.7717$
apart in space, $9.6824$ on the flattened page.
Neither view is *wrong*; they answer different questions — and here is
the discipline, the concept this unit's last problems grade:

| Question about the data | Trust | Because |
|---|---|---|
| "Are these two points genuinely similar (neighbors)?" | neighbor-preserving view (here: unrolled) | local structure is what it keeps ($0.93$ vs $0.59$) |
| "How far apart are these two regions, really?" | PCA view — or better, distances in the original space | projections never exaggerate; nonlinear maps do ($4.3\times$) |
| "What does the horizontal axis *mean*?" | PCA only | PCA axes are stated feature blends (rows of $V^{\mathsf T}$); layout axes mean nothing |
| "How much of the variance did the picture keep?" | PCA only | evr exists only for projections |

**The habit: decide what question you are asking *before* choosing
which map to believe.**

### Checkpoint 6

1. Why is the PCA view's worst-case stretch exactly $1$ (up to float
   noise) rather than, say, $0.9$?
2. The ends-of-the-S pair: which view reports their relationship
   *as-the-crow-flies*, and which reports it *as-the-ant-walks*?
   Which is "right"?
3. A colleague wants to know whether two specific songs in a
   $100$-D feature space are near-duplicates, using only 2-D
   pictures. Which row of the table applies, and what does it
   prescribe?

## 7. Worked Exam-Style Example: Multiple Choice

The concept register, worked in full.

---

**Problem.**
A 2-D map of a $64$-dimensional dataset is produced by a
neighbor-graph layout method (UMAP-style, as taught: $k$-NN graph in
the original space, laid out so graph neighbors stay close).
Which conclusion is *best supported* by the map alone?

A. Two points lying in the same tight clump are likely near neighbors
in the $64$-D space.

B. Clump 1 is three times as far from clump 2 as from clump 3, so its
members are three times as dissimilar.

C. The map's horizontal axis is the dataset's most important feature.

D. Clump 2 looks twice as wide as clump 3, so its members are twice as
spread out.

E. Rerunning the method will reproduce this exact layout.

*Reasoning is not required (but we reason anyway).*

---

**Work the options against the stated facts.**
B reads a *global distance ratio* off the page — stated fact 2's
casualty, first sentence.
C assigns meaning to an axis — stated fact 3 (no units, no feature
recipes).
D reads *apparent size* quantitatively — stated fact 2 again
(densities and sizes distort).
E contradicts stated fact 4 (settings and randomness change layouts).
A is stated fact 1 read correctly — tight on-page neighborhoods
reflect genuine neighbor relations, which is the one thing the method
is built to preserve.
Answer: **A**.

Note the exam shape: four options that each violate exactly one stated
fact, one option that *is* a stated fact.
Learn the facts as claims-with-scope and these become sight-reads.

### Checkpoint 7

1. Which stated fact does each wrong option violate? (Match B, C, D,
   E to facts 2, 2, 3, 4.)
2. Rewrite option A so that it *overclaims* — turning the supported
   local statement into an unsupported global one.

## 8. Common Pitfalls

**Pitfall 1 — measuring the picture (quiet).**
Any distance you compute *in map coordinates* — a ruler on the page —
is a statement about the *map*, not the data, unless the map's
guarantee covers it.
The S-ribbon numbers to keep in your pocket: a nonlinear flattening
stretched a $1.17$ gap to $5.02$; a faithful-variance projection
compressed a $1.95$ gap to $0.02$.
Both pictures were "good"; both rulers lied.

**Pitfall 2 — asking a view the wrong question (quiet).**
The $91\%$-evr PCA view *scored* $0.59$ on neighborhoods; the
neighborhood-perfect unrolled view stretched global distances
$4.3\times$.
Neither number is a flaw — each is the price of the other view's
virtue.
The failure mode is using one view for both questions.

**Pitfall 3 — trusting cluster counts from one picture (quiet).**
A fold can merge two far stretches into one apparent clump (the PCA
view did); a layout's randomness can split or shuffle apparent groups
(stated fact 4).
Structural claims — "there are three groups" — deserve a check that
does not depend on any single 2-D picture: the neighbor graph itself,
metrics in the original space, or (in practice p17) connected
components of the $k$-NN graph.

**Pitfall 4 — forgetting the one-sided guarantee's direction
(quiet).**
"Far in the PCA view ⇒ far in reality" is valid.
"Close in the PCA view ⇒ close in reality" is the false-neighbor
trap.
Students flip this under time pressure; the ribbon's pair
$(0.0231 \to 1.9532)$ is the counterexample to memorize.

### Checkpoint 8

1. A report claims "the outlier is 4 map-centimeters from its
   cluster, i.e. wildly anomalous," based on a UMAP-style layout.
   Which pitfall(s), and which stated fact?
2. Which *single* direction of inference about distances does the PCA
   view license, and in one clause, why?

## Exam Connections

How this session's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic tables — no real test text here):

- The **ML-concepts cluster** includes a multiple-choice item on UMAP
  whose correct option is stated fact 1 verbatim — *prioritizes
  preserving local neighborhood structure over global distances* —
  with distractors violating the other stated facts (exact distance
  preservation, linearity, PCA's reconstruction objective).
  Section 7's worked MC is that item's register.
- **Dimensionality-reduction vocabulary** (projection, variance
  explained, low-rank, local-vs-global trade-offs) threads the exam's
  big linear-algebra arc; the concept questions assume you can *say
  what a map preserves* without running anything.
- **Grading signals**: concept MCs are "reasoning not required" —
  sight-read speed comes from holding the stated facts as
  claims-with-scope, exactly how this session drilled them.

## Going Deeper

Optional forward pointers along the course map — nothing here is
needed for this unit's practice:

- **`C10-competition-craft`** — the course's final unit, where
  everything C1–C9 built gets exercised under competition conditions:
  the graded-notebook contract, hidden-test discipline,
  metric-driven iteration.
  This unit's habit — *know what question you are asking before
  choosing the tool* — is the mindset that unit turns into a
  workflow.
- **Manifold learning, beyond the concept** — the neighbor-graph idea
  taught here as stated fact is a whole field (UMAP, t-SNE, Isomap,
  and kin). If you later install such a library, this session's
  metrics — $k$-NN preservation and worst-case stretch — are exactly
  how to audit what its pictures do to *your* data.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. For $t > 0$: $x' = \cos t$, $z' = -\sin t$, so
   $x'^2 + z'^2 = 1$; likewise for $t < 0$ with $z' = \sin t$.
   Unit speed means arc length $=$ parameter change.
2. About $|4.0 - 1.0| = 3.0$ — unit speed makes the parameter gap the
   ribbon distance (the same-$y$ condition means no across-ribbon
   contribution).
3. Delete $x$: the S lives in the $xz$ profile, and without $x$ the
   two hooks and the middle collapse onto each other (which is
   exactly what the PCA view ends up doing).

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $v_3 \approx -e_x$: the view keeps (essentially) $y$ and $z$ and
   discards $x$ — looking at the S from above, so branches at
   different $x$ but similar $(y, z)$ land on top of each other.
2. No — width is uniform filler. Evr ranks variance, not
   interestingness: the largest-variance direction here is the least
   structured one.
3. Their separation directions were the *dominant variance*
   directions (and the structure was linear/flat); the ribbon's
   defining direction is its *smallest*-variance coordinate, and its
   structure is curved.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. "Far in the view" is evidence — of genuine farness (distances
   never grow under projection). "Close in the view" is *not*
   evidence of closeness — it may be a fold artifact.
2. Without the mask, the argmin returns a genuinely-adjacent pair
   (view distance near 0 because true distance is near 0) — true
   neighbors, not false ones. The mask restricts the search to pairs
   the ribbon says are far.
3. Yes — that is the *other* mismatch direction: the hook tips curl
   back toward the middle, so look near $|t| \approx 4.7$ against
   $t \approx 0.3$ (Section 6 finds exactly such a pair: $1.1667$
   apart in 3-D, five ribbon-units apart).

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Typical nearest-neighbor distances on a $600$-point,
   $9.4 \times 6$ sheet are a few tenths; the fold gap is
   $\approx 1.9$ ($\approx 2$ in $x$ alone). Nearest-neighbor lists
   fill up with within-sheet points long before any across-fold
   point qualifies.
2. Stated fact 2 (global distances, densities, sizes carry no
   quantitative meaning).
3. Each PCA axis is a stated unit-vector recipe over the original
   features (a row of $V^{\mathsf T}$), with an evr attached; a
   layout axis has neither.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Every point's nearest "neighbor" would otherwise be itself
   (distance $0$), wasting one of the $k$ slots and inflating every
   overlap by one guaranteed hit.
2. Chance level $\approx k/(n-1) = 10/599 \approx 0.017$: a random
   view's neighbor lists are $k$ draws from $n - 1$ candidates with
   no relation to the true lists.
3. The noise ($\sigma = 0.03$ in all three coordinates): points near
   the edge of a neighborhood swap in and out between the noisy 3-D
   ranking and the clean $(t, y)$ ranking.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Distances never grow (ratio $\le 1$), and any pair aligned with
   the kept plane (e.g. two points differing mainly in $y$) keeps
   its distance essentially exactly (ratio $\to 1$); the max over
   all pairs is therefore $1$.
2. 3-D/PCA reports as-the-crow-flies ($3.77$); the unrolled view
   reports as-the-ant-walks ($9.68$). Neither is "right" — they are
   answers to different questions (chord vs path).
3. Row 1 (near-duplicate = neighbor question): trust a
   neighbor-preserving view — or better, skip pictures and compute
   the distance in the original $100$-D space directly.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. B → fact 2 (global distances); C → fact 3 (axes meaningless);
   D → fact 2 (sizes/densities distort); E → fact 4 (randomness and
   settings).
2. E.g. "Two points in the same tight clump are likely near
   neighbors, *and the clump's distance from other clumps measures
   how different they are*" — the added clause is exactly fact 2's
   casualty.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Pitfall 1 (measuring the picture) and pitfall 2 (wrong question
   for the view); stated fact 2 — map distances carry no
   quantitative meaning.
2. Far-in-view ⇒ far-in-reality — because an orthonormal projection
   can only shrink or preserve distances, never stretch them.

</details>